# Pipeline de Limpieza de ECYL

El primer paso es importar las librerías que vamos a utilizar

In [2]:
import requests
import os
import pandas as pd
import json
from pathlib import Path

In [3]:
api_key = os.environ.get("API_KEY")
headers = {
    "Authorization": f"Apikey {api_key}",
    "Content-Type": "application/json"
}

Prueba de que se ha podido acceder a la API

In [4]:
url = "https://analisis.datosabiertos.jcyl.es/api/explore/v2.1/catalog/datasets/ofertas-de-empleo/records/"

limite = 100
offset = 0
todos_los_registros = []
total = None

while total is None or offset < total:
    respuesta = requests.get(
        url,
        params={"lang": "es", "limit": limite, "offset": offset},
        timeout=30
    )
    respuesta.raise_for_status()

    pagina = respuesta.json()
    total = pagina["total_count"]
    registros = pagina["results"]

    if not registros:
        break

    todos_los_registros.extend(registros)
    offset += len(registros)

print(f"Descargados {len(todos_los_registros)} de {total} registros")

with open("respuestas.json", "w", encoding="utf-8") as archivo:
    json.dump(
        {"total_count": total, "results": todos_los_registros},
        archivo,
        ensure_ascii=False,
        indent=2
    )

Descargados 1027 de 1027 registros


# Carga de datos

Mostramos que se han obtenido correctamente

In [5]:
with open("respuestas.json", "r", encoding="utf-8") as archivo:
    datos = json.load(archivo)

df = pd.json_normalize(datos["results"])

print(df.shape)  # filas y columnas
df.head()

(1027, 16)


,titulo,provincia,fecha_publicacion,descripcion,provincia_alternativa,fuente_contenido,localidad_asset,localidad_asset0,localidad_asset2,documentos_asociados,identificador,actualizacionmetadatos,enlace_al_contenido,geografico.lon,geografico.lat,geografico
0,"COCINEROS, EN GENERAL",Palencia,2026-10-12,"<p>COCINEROS, PARA DUEÑAS. LABORAL INDEFINIDO ...",None,ECYL,1179298081851,Dueñas,34.0,None,1285689059800,2026-10-12,https://datosabiertos.jcyl.es/web/jcyl/Empleo/...,-4.546755,41.877316,NaN
1,"COCINEROS, EN GENERAL",Soria,2026-10-11,<p>OFERTA 21165. HOTEL EN EL BURGO DE OSMA (SO...,None,ECYL,1179469126353,Burgo de Osma-Ciudad de Osma,42.0,None,1285689059772,2026-10-12,https://datosabiertos.jcyl.es/web/jcyl/Empleo/...,-3.075403,41.588287,NaN
2,TERAPEUTAS OCUPACIONALES,León,2026-10-11,"<p>Para más información acuda a <a href=""http:...",None,ECYL,1179297955047,Soto de la Vega,24.0,None,1285689059856,2026-10-12,https://datosabiertos.jcyl.es/web/jcyl/Empleo/...,-5.882215,42.332270,NaN
3,"PEONES DE LA INDUSTRIA DE LA ALIMENTACIÓN, BEB...",Salamanca,2026-10-10,"<p>Para más información acuda a <a href=""http:...",None,ECYL,1179298266466,Ledrada,37.0,None,1285689059884,2026-10-12,https://datosabiertos.jcyl.es/web/jcyl/Empleo/...,-5.720769,40.469257,NaN
4,Técnico/a Medio de Contabilidad para Diputació...,Palencia,2026-10-09,<p><strong>Convoca: </strong> Diputación de Pa...,None,Organismo oficial,None,None,NaN,None,1285688752400,2026-10-09,https://datosabiertos.jcyl.es/web/jcyl/Empleo/...,NaN,NaN,NaN


# Transformaciones

### Fechas

In [10]:
print(df["fecha_publicacion"].dtype)
print(df["actualizacionmetadatos"].dtype)

object
object


Las columnas de fechas_publicacio y actualizacionmetadatos contienen fechas que están en formato de cadenas de texto. Las transformamos a datetime para tratarlas de forma más sencilla y permitir operaciones con los tiempos.

Se utiliza el formato ISO 8601 y se establece utc como zona horaria.

In [11]:
for columna in ["fecha_publicacion", "actualizacionmetadatos"]:
    df[columna] = pd.to_datetime(df[columna], format="ISO8601", errors="raise", utc=True)

### Valores duplicados

Guardamos los datos en un csv en la carpeta silver

In [6]:
carpeta = Path(r"C:\Users\Alejandro\Desktop\DataScience\Proyecto Evolve\Proyecto Grupal Evolve\data\silver\ECYL")
carpeta.mkdir(parents=True, exist_ok=True)

df.to_csv(carpeta / "ofertas_limpias.csv", index=False, encoding="utf-8-sig")